# 🍄 Mario RL — CNN Training (PPO + CnnPolicy)

Trains a PPO agent on **Super Mario Bros World 1-1** using **pixel observations** (84×84 grayscale, 4-frame stack).

| Setting | Value |
|---|---|
| Algorithm | **PPO** (Proximal Policy Optimization) |
| Policy network | **CnnPolicy** (NatureCNN backbone — learns from visual frames) |
| Observation | 84×84 grayscale, 4-frame stack → uint8 (84×84×4) |
| Level | World 1-1 (flag-locked) |
| Episode ends | Mario dies **or** reaches the flag |
| Training device | **GPU** (CNN convolutions benefit heavily from CUDA) |

> **Runtime:** Use a **GPU** runtime (A100 or T4). The NatureCNN convolutions run much faster on GPU than CPU.

### How the Pixel Pipeline Works
```
Retro Screen (224×240×3 RGB)
    ↓  PixelPreprocess:  grayscale + resize
    84×84×1  uint8
    ↓  ActionRepeat:  step 4 frames
    ↓  FrameStack:  keep last 4 frames
    84×84×4  uint8
    ↓  VecTransposeImage (SB3 auto)
    4×84×84  → NatureCNN → PPO
```

## 1. Mount Google Drive
All checkpoints and logs are saved to Drive so they survive runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Adjust these paths to match your Drive layout
DRIVE_ROOT   = '/content/drive/MyDrive/mario_rl'
MODEL_DIR    = f'{DRIVE_ROOT}/models'
LOG_DIR      = f'{DRIVE_ROOT}/runs'
ROM_DIR = '/content/mario-rl-ram/roms'
VIDEO_DIR    = f'{DRIVE_ROOT}/videos/cnn'

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR,   exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)
print('Drive mounted. Directories ready.')

## 2. Clone Repo & Install

In [ ]:
# Clone (or update) the repo
import os
if not os.path.isdir('/content/mario-rl-ram'):
    !git clone https://github.com/hbofz/mario-rl-ram.git /content/mario-rl-ram
else:
    !git -C /content/mario-rl-ram pull

%cd /content/mario-rl-ram
!pip install -e . -q
print('Install complete. Restart runtime if prompted.')

## 3. Verify GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'✅ GPU available: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. CNN training will be slow. Consider switching to a GPU runtime.')

## 4. Import ROM

Imports the ROM from the repo's `roms/` folder. If you keep your ROM in Drive instead, change `ROM_DIR` above.

In [ ]:
!python -m stable_retro.import {ROM_DIR}

## 5. Smoke Test
Quick sanity check for the **pixel** pipeline — should print observation shape `(84, 84, 4)` and action space.

In [ ]:
!mario-smoke --state Level1-1 --obs-mode pixel --steps 300 --reward-mode base

## 6. Training Configuration

> **CNN hyperparameter note:** Compared to RAM training, we use:
> - **fewer parallel envs** (`n_envs=8`) — pixel preprocessing is heavier per step
> - **shorter rollouts** (`n_steps=128`) — pixel frames consume more memory
> - **smaller batch** (`batch_size=512`) — fits GPU VRAM
> - **fewer total steps** (`5M`) — CNN learns faster per step due to richer input

In [ ]:
# ── Training hyperparameters ──────────────────────────────────────────────────
RUN_NAME      = 'cnn-1-1'          # name of this training run
TOTAL_STEPS   = 5_000_000          # total environment steps
N_ENVS        = 8                  # parallel emulator workers (fewer due to pixel cost)
N_STEPS       = 128                # steps per env per rollout (shorter for memory)
BATCH_SIZE    = 512                # PPO minibatch size
N_EPOCHS      = 4                  # PPO update epochs per rollout
LEARNING_RATE = 2.5e-4             # Adam learning rate
GAMMA         = 0.99               # discount factor
GAE_LAMBDA    = 0.95               # GAE lambda
ENT_COEF      = 0.01               # entropy coefficient (exploration)
CLIP_RANGE    = 0.1                # PPO clip range (tighter for pixel policy)
CKPT_FREQ     = 250_000            # save checkpoint every N steps

# ── Environment settings ──────────────────────────────────────────────────────
STATE         = 'Level1-1'         # which level to train on
OBS_MODE      = 'pixel'            # 'pixel' for this notebook
ACTION_MODE   = 'mario'            # curated 11-action discrete set
REWARD_MODE   = 'smart'            # per-level reward profile
ACTION_REPEAT = 4                  # repeat each action for N frames
SINGLE_LIFE   = True               # end episode on first life lost
FLAG_LOCK     = True               # end episode when flag is reached

print(f'Run: {RUN_NAME}')
print(f'Algorithm: PPO  |  Policy: CnnPolicy (NatureCNN on pixel frames)')
print(f'Observation: 84×84 grayscale, 4-frame stack')
print(f'Level: {STATE}  |  Total steps: {TOTAL_STEPS:,}')
print(f'Envs: {N_ENVS}  |  Batch: {BATCH_SIZE}  |  Checkpoint every: {CKPT_FREQ:,} steps')

## 7. Train
Auto-resume is **on** by default — if Colab disconnects, re-run this cell and training continues from the latest checkpoint.

In [ ]:
!mario-train \
  --obs-mode        {OBS_MODE} \
  --state           {STATE} \
  --flag-lock \
  --algo            ppo \
  --timesteps       {TOTAL_STEPS} \
  --n-envs          {N_ENVS} \
  --n-steps         {N_STEPS} \
  --batch-size      {BATCH_SIZE} \
  --n-epochs        {N_EPOCHS} \
  --learning-rate   {LEARNING_RATE} \
  --gamma           {GAMMA} \
  --gae-lambda      {GAE_LAMBDA} \
  --ent-coef        {ENT_COEF} \
  --clip-range      {CLIP_RANGE} \
  --checkpoint-freq {CKPT_FREQ} \
  --action-mode     {ACTION_MODE} \
  --reward-mode     {REWARD_MODE} \
  --action-repeat   {ACTION_REPEAT} \
  --single-life \
  --run-name        {RUN_NAME} \
  --model-dir       {MODEL_DIR} \
  --log-dir         {LOG_DIR} \
  --device          auto

## 8. Monitor Training with TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

## 9. Evaluate & Record Videos

In [ ]:
EVAL_MODEL    = f'{MODEL_DIR}/{RUN_NAME}/final_model.zip'
EVAL_EPISODES = 5

!mario-eval \
  --model     {EVAL_MODEL} \
  --obs-mode  {OBS_MODE} \
  --state     {STATE} \
  --episodes  {EVAL_EPISODES} \
  --video-dir {VIDEO_DIR} \
  --no-single-life \
  --no-flag-lock

## 10. Evaluate a Specific Checkpoint
Compare early vs. late training to see how the visual policy evolved.

In [ ]:
CHECKPOINT = f'{MODEL_DIR}/{RUN_NAME}/ppo_2500000_steps.zip'  # ← change step count

!mario-eval \
  --model     {CHECKPOINT} \
  --obs-mode  {OBS_MODE} \
  --state     {STATE} \
  --episodes  3 \
  --video-dir {VIDEO_DIR}/checkpoint_2_5M \
  --no-single-life \
  --no-flag-lock

## 11. Inspect What the CNN "Sees"
Visualize the preprocessed grayscale frames the policy actually receives.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mario_rl import make_mario_env

env = make_mario_env(
    state='Level1-1',
    obs_mode='pixel',
    action_mode='mario',
    reward_mode='base',
    flag_lock=False,
    single_life=False,
    monitor=False,
)

obs, _ = env.reset()
# Take a few random steps to get a non-trivial frame
for _ in range(20):
    obs, _, done, _, _ = env.step(env.action_space.sample())
    if done:
        obs, _ = env.reset()

# obs shape: (84, 84, 4) — 4 stacked grayscale frames
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle('CNN Input: 4 stacked grayscale frames (most recent = rightmost)', fontsize=12)
for i, ax in enumerate(axes):
    ax.imshow(obs[:, :, i], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'Frame t-{3-i}')
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{VIDEO_DIR}/cnn_input_frames.png', dpi=120)
plt.show()
env.close()
print(f'Observation shape: {obs.shape}  dtype: {obs.dtype}')

## 12. List Saved Checkpoints

In [ ]:
import os, glob
ckpts = sorted(glob.glob(f'{MODEL_DIR}/{RUN_NAME}/*.zip'))
print(f'Found {len(ckpts)} checkpoint(s):')
for c in ckpts:
    size_mb = os.path.getsize(c) / 1e6
    print(f'  {os.path.basename(c)}  ({size_mb:.1f} MB)')